# 149 — Serving online, batch y streaming

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


## Solución de referencia

**Ejercicio 1.**
a) **Batch** semanal: nadie espera en línea y la decisión tolera días de edad.
b) **Online**: el usuario espera la publicación; latencia percibida manda.
c) **Online** con presupuesto duro de 300 ms: hay una transacción esperando la decisión.
d) **Streaming**: la feature es un agregado por ventana de eventos que debe estar fresca
*antes* de la petición; el patrón completo c+d es el híbrido clásico «features en
streaming + inferencia online».

**Ejercicio 2.** Throughput por réplica = 6 / 0.120 = **50 rps**. Demanda con margen =
90 × 1.4 = 126 rps → 126/50 = 2.52 → **3 réplicas**. Con 5 réplicas: 5 × 50 =
**250 rps** máximos teóricos (en la práctica menos: el lote de 6 no siempre se llena, y
al bajar el lote cae el throughput — acoplamiento lote-latencia del batching dinámico).

**Ejercicio 3.** La media (55 ms) está dominada por la mayoría rápida y es ciega a la
cola: el 1 % de peticiones a 900 ms es invisible en el promedio pero define la
experiencia de quien la sufre. Con 10 llamadas independientes:
`P(al menos una en el peor 1 %) = 1 − 0.99¹⁰ ≈ 0.096` → **~9.6 %** de las páginas tocan
la cola. Con 100 llamadas sería ~63 %: por eso los sistemas de fan-out alto viven o
mueren por el p99 (argumento central de *The Tail at Scale*).

**Ejercicio 4.** En `evidence`, los tiempos por operación juegan el papel de latencias;
los conteos por ventana, el de throughput; las marcas de tiempo de generación de datos,
el de frescura. `limitations` recuerda que son magnitudes simuladas y deterministas.


In [ ]:
result = run_lab("observability", seed=149)
assert result["kind"] == "observability"
assert result["evidence"]
show(result)


In [ ]:
import math

# Ejercicio 2
rps_pico, lote, lat_s, margen = 90, 6, 0.120, 1.40
tput_replica = lote / lat_s
replicas = math.ceil(rps_pico * margen / tput_replica)
print(f"throughput/réplica = {tput_replica:.0f} rps")
print(f"réplicas necesarias = {replicas}")
print(f"throughput con 5 réplicas = {5 * tput_replica:.0f} rps")

# Ejercicio 3
p = 1 - 0.99 ** 10
print(f"P(página de 10 llamadas toca el peor 1 %) = {p:.3f}")


## Reflexión

1. Una página compone 20 llamadas a servicios con p99 = 100 ms cada uno: ¿por qué la latencia de la página será mala aunque «solo» el 1 % de las llamadas sea lenta, y qué técnica del serving online lo mitiga?
2. ¿Qué requisito habría que cambiar en el ejemplo del e-commerce para que streaming fuera la elección correcta en lugar del híbrido batch+online?
3. Si duplicas el tamaño de lote del batching dinámico, ¿qué pasa con el throughput y con el p95, y cómo decidirías el punto de operación?
